[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giuliofranzese/Cajal/blob/main/functional_kl/code/notebooks/tutorial_gm_pretrained.ipynb)

# GM Dataset — Functional KL Divergence Estimation (Checkpoints)

This notebook runs the **Gaussian Mixture (GM)** benchmark experiment on Google Colab.

It estimates KL divergence between two 1D trajectory distributions (A vs B) using Functional Flow Matching (FFM) with the MINO-T architecture.

**Pipeline:**  Copy pre-trained checkpoint → Estimate KL divergence via Fourier basis projection

**Data shape:** 50000 samples × 128 timesteps × 1 dimension

## a) With Github repo

If you have access to the public repo, run the following command and skip (b).

In [ ]:
!git clone https://github.com/giuliofranzese/Cajal.git

## b) With Google Drive

Run the following command if you have the folder in your google drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0. Setting up the environment

In [ ]:
# change it with the correct path to the notebooks

%cd "path_to_notebooks" 

In [ ]:
!ls

In [ ]:
!pip install \
einops==0.8.2 \
gpytorch==1.15.2 \
kappamodules==0.1.112 \
neuraloperator==2.0.0 \
pytorch-lightning==2.6.1 \
setproctitle==1.3.7 \
torch-geometric==2.7.0 \
torchcfm==1.0.7 \
torchdiffeq==0.2.5 \
wandb==0.25.1

## 1. Imports & Configuration

In [ ]:
import os, sys
import numpy as np
import torch

# Find repo root
if os.path.isdir("util"):
    REPO_ROOT = os.path.abspath(".")
else:
    REPO_ROOT = os.path.abspath("..")

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from util.util import *

# --- Set up the hardware device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Currently running on: {device}")

In [ ]:
# Configuration
CHECKPOINT_EPOCH = 307           # epoch of the pre-trained checkpoint

os.chdir(REPO_ROOT)
print("Working directory:", os.getcwd())

# Paths (relative to repo root)
SCRIPTS_DIR = "scripts"
CONFIG      = "configs/gm.yaml"
DATA_DIR    = "data/GM"
LOG_DIR     = "log/GM"

M = 128  # grid size

## 2. Copy Pre-trained Checkpoint

The checkpoint from `checkpoints/GM/epoch_307.pt` is placed in the expected log directory so that training is skipped and FKL estimation runs directly.

In [ ]:
import shutil

ckpt_src = f"checkpoints/GM/epoch_{CHECKPOINT_EPOCH}.pt"
ckpt_dst_dir = f"{LOG_DIR}/A_vs_B/ckpt"
ckpt_dst = f"{ckpt_dst_dir}/epoch_{CHECKPOINT_EPOCH}.pt"

os.makedirs(ckpt_dst_dir, exist_ok=True)
shutil.copy(ckpt_src, ckpt_dst)
print(f"Checkpoint ready: {ckpt_dst}")

## 3. Visualization

In [ ]:
data_A = np.load("data/GM/X1_data_A_128.npy")
data_B = np.load("data/GM/X1_data_B_128.npy")

print(f"A shape: {data_A.shape}")
print(f"B shape: {data_B.shape}")

In [ ]:
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})

N_LINES = 200
ALPHA = 0.15
LW = 0.6

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

tau = np.linspace(0, 1, M)

rng = np.random.default_rng(42)
n_show = min(N_LINES, data_A.shape[0], data_B.shape[0])
idx_A = rng.choice(data_A.shape[0], size=n_show, replace=False)
idx_B = rng.choice(data_B.shape[0], size=n_show, replace=False)

for i in idx_A:
    axes[0].plot(tau, data_A[i, :, 0] if data_A.ndim == 3 else data_A[i, :],
                color="tab:blue", alpha=ALPHA, lw=LW)
axes[0].set_title("Distribution A")
axes[0].set_xlabel("$\\tau$")
axes[0].set_ylabel("Value")

for i in idx_B:
    axes[1].plot(tau, data_B[i, :, 0] if data_B.ndim == 3 else data_B[i, :],
                color="tab:orange", alpha=ALPHA, lw=LW)
axes[1].set_title("Distribution B")
axes[1].set_xlabel("$\\tau$")

plt.tight_layout()
plt.show()

## 4. Compute FKL (A vs B)

Loads the pre-trained checkpoint and runs FKL estimation (training is skipped).

In [ ]:
# Set the GPU variable globally for the notebook
%env CUDA_VISIBLE_DEVICES=0

# Change the directory
%cd {SCRIPTS_DIR}

# Run the script
!python main.py \
    -c ../{CONFIG} \
    --set data_params.X1.A_path.{M}=../{DATA_DIR}/X1_data_A_{M}.npy \
    --set data_params.X1.B_path.{M}=../{DATA_DIR}/X1_data_B_{M}.npy \
    --set trainer_params.sfolder={LOG_DIR}/A_vs_B \
    --set trainer_params.epochs_selected={CHECKPOINT_EPOCH}

## 5. Inspect Results

Load the output config files which contain the estimated KL divergence values.

In [ ]:
import yaml
import os
from pathlib import Path

print(f"Current notebook directory: {os.getcwd()}")
print(f"LOG_DIR variable is currently: {LOG_DIR}")

run_dir = Path("../log/GM/A_vs_B").resolve()
print(f"Looking for files in absolute path: {run_dir}")

result_files = sorted(run_dir.glob("config_kl_FINAL_*.yaml"))
print(f"Files found: {[f.name for f in result_files]}")
print("-" * 44)

if not result_files:
    print("No KL result files found yet. The path above is incorrect.")
else:
    with open(result_files[-1], "r") as f:
        cfg = yaml.safe_load(f)

    kl = cfg.get("kl_result", {}).get(1, {})
    fwd = kl.get("forward_KL", "N/A")
    rev = kl.get("reverse_KL", "N/A")

    fwd_str = f"{fwd:.4f}" if isinstance(fwd, (int, float)) else str(fwd)
    rev_str = f"{rev:.4f}" if isinstance(rev, (int, float)) else str(rev)

    print(f"{'Comparison':<16} {'Forward KL':>12} {'Reverse KL':>12}")
    print("-" * 44)
    print(f"{'A vs B':<16} {fwd_str:>12} {rev_str:>12}")

## 6. View Generated Trajectory Plots

In [ ]:
!sudo apt-get update && sudo apt-get install poppler-utils

In [ ]:
from IPython.display import display, Image
from pathlib import Path
import subprocess
import os

run_dir = Path("../log/GM/A_vs_B/imgs").resolve()
print(f"Looking for PDFs in absolute path: {run_dir}")

if not run_dir.exists():
    print("Directory not found! Double-check that the script actually created an 'imgs' folder.")
else:
    pdf_files = sorted(run_dir.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF files. Processing...\n" + "-"*40)

    if not pdf_files:
        print("No PDFs found in this folder.")

    for pdf in pdf_files:
        if "energyspectrum" in pdf.name:
            continue

        png = pdf.with_suffix(".png")

        try:
            subprocess.run(["pdftoppm", "-png", "-r", "150", "-singlefile", str(pdf), str(png.with_suffix(""))], check=True)
            print(f"\n{pdf.name}")
            display(Image(filename=str(png)))

        except FileNotFoundError:
            print("ERROR: The Linux tool 'pdftoppm' is not installed on this Colab VM!")
            break